Robustness Check 0: Pre-trend check

In [3]:
# ==========================================================
#  PRE-TREND TEST: k = -2 (event_m2_dose)
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load the stacked datasets saved from main regression
# (reuse all_stacks produced earlier)
pretrend_results = []

for stack in all_stacks:
    ind = stack["industry"].iloc[0]

    # Identify the pre-trend variable
    pre_var = "event_m2_dose"
    if pre_var not in stack.columns:
        continue

    # Keep only observations with variation
    if stack[pre_var].sum() == 0:
        continue

    formula = f"""
    employment_rate ~
        {pre_var}
        + C(province) + C(date)
    """

    model = smf.ols(formula, data=stack).fit(
        cov_type="cluster",
        cov_kwds={"groups": stack["province"]}
    )

    wald = model.wald_test(f"{pre_var} = 0", scalar=True)

    pretrend_results.append({
        "industry": ind,
        "pre_k": -2,
        "wald_stat": float(wald.statistic),
        "p_value": float(wald.pvalue),
        "n_obs": int(model.nobs)
    })

pretrend_df = pd.DataFrame(pretrend_results)

print("\n=== Short-run Pre-trend Test (k = -2) ===")
print(pretrend_df.sort_values("p_value").to_string(index=False))



=== Short-run Pre-trend Test (k = -2) ===
                                           industry  pre_k  wald_stat  p_value  n_obs
                              Public administration     -2     6.0721   0.0137   4209
                  Health care and social assistance     -2     3.4832   0.0620   4209
                             Goods-producing sector     -2     2.3942   0.1218   4209
                     Transportation and warehousing     -2     1.9279   0.1650   4209
                              Total, all industries     -2     1.2792   0.2580   4209
                          Services-producing sector     -2     1.0778   0.2992   4209
                                       Construction     -2     0.4718   0.4922   4209
      Other services (except public administration)     -2     0.4173   0.5183   4209
                               Educational services     -2     0.3425   0.5584   4209
                                          Utilities     -2     0.3266   0.5677   4125
           

In [6]:
# ==========================================================
#  CLEAN STACKED DID
#  Baseline = t0 (policy month)
#  Joint pre-trend test: k = -1, -2
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# -----------------------------
# 0. PATH
# -----------------------------
base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
DATA = fr"{base}\14100355.csv"

# -----------------------------
# 1. Read and clean data
# -----------------------------
raw = pd.read_csv(DATA)

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"])

raw["province"] = raw["province"].astype(str).str.strip()
raw["industry"] = raw["industry"].astype(str).str.strip()

raw["min_wage"] = pd.to_numeric(raw["min_wage"], errors="coerce")
raw["employment_rate"] = pd.to_numeric(raw["employment_rate"], errors="coerce")

# Monthly index
raw["m_id"] = raw["date"].dt.year * 12 + raw["date"].dt.month

# -----------------------------
# 2. Construct baseline wage (pre-2023, policy series only)
# -----------------------------
policy = raw.dropna(subset=["min_wage"]).copy()
policy = policy.sort_values(["province", "date"])

base_wage = (
    policy.loc[policy["date"] < "2023-01-01", ["province", "date", "min_wage"]]
          .groupby("province", as_index=False)
          .tail(1)
          .rename(columns={"min_wage": "wage_base"})
          [["province", "wage_base"]]
)

raw = raw.merge(base_wage, on="province", how="left")

raw["dlogW"] = np.log(raw["min_wage"]) - np.log(raw["wage_base"])

# -----------------------------
# 3. Regression-ready dataframe
# -----------------------------
df = raw.dropna(subset=["employment_rate", "min_wage", "dlogW"]).copy()
df = df.sort_values(["province", "date"])

# -----------------------------
# 4. Identify 2023+ MW events (province level)
# -----------------------------
df["dW"]   = df.groupby("province")["min_wage"].diff()
df["lagW"] = df.groupby("province")["min_wage"].shift(1)
df["dose"] = (df["dW"] / df["lagW"]) * 100

events = (
    df.loc[
        (df["dose"] > 0) & (df["date"] >= "2023-01-01"),
        ["province", "date", "dose", "m_id"]
    ]
    .drop_duplicates()
    .rename(columns={"date": "t0", "m_id": "m0"})
    .reset_index(drop=True)
)

print(f"Identified 2023+ MW increase events: {len(events)}")

# -----------------------------
# 5. Event window
# -----------------------------
L, R = 2, 3   # leads/lags
BASE_K = 0    # ★ baseline = t0

# -----------------------------
# 6. Global clean-control indicator
# -----------------------------
df["in_any_window"] = 0

for prov, g in events.groupby("province"):
    m0_list = g["m0"].tolist()
    idx = df["province"].eq(prov)
    m = df.loc[idx, "m_id"].values
    hit = np.zeros_like(m, dtype=bool)
    for m0 in m0_list:
        hit |= (m >= m0 - L) & (m <= m0 + R)
    df.loc[idx, "in_any_window"] = hit.astype(int)

# -----------------------------
# 7. Build stacked samples by industry
# -----------------------------
industries = sorted(df["industry"].unique())
all_stacks = []

for ind in industries:
    dfi = df[df["industry"] == ind].copy()
    stack_list = []

    for ev in events.itertuples(index=False):
        ev_prov = ev.province
        m0      = ev.m0
        dose    = ev.dose

        # ---- Treated ----
        treated = dfi[dfi["province"] == ev_prov].copy()
        treated["EventTime"] = treated["m_id"] - m0

        treated = treated[
            (treated["EventTime"] >= -L) &
            (treated["EventTime"] <= R)
        ].copy()

        # Remove overlaps within province
        other_m0 = events.loc[
            (events["province"] == ev_prov) & (events["m0"] != m0),
            "m0"
        ].tolist()

        if other_m0:
            m = treated["m_id"].values
            overlap = np.zeros_like(m, dtype=bool)
            for om0 in other_m0:
                overlap |= (m >= om0 - L) & (m <= om0 + R)
            treated = treated[~overlap]

        # Event-time × dose (baseline = k=0 omitted)
        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            var = f"event_{'m'+str(abs(k)) if k < 0 else 'p'+str(k)}_dose"
            treated[var] = ((treated["EventTime"] == k).astype(int)) * dose

        if treated.empty:
            continue

        # ---- Controls ----
        controls = dfi[
            (dfi["province"] != ev_prov) &
            (dfi["in_any_window"] == 0)
        ].copy()

        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            var = f"event_{'m'+str(abs(k)) if k < 0 else 'p'+str(k)}_dose"
            controls[var] = 0.0

        stack_list.append(pd.concat([treated, controls], ignore_index=True))

    if not stack_list:
        continue

    stack = pd.concat(stack_list, ignore_index=True)
    stack["industry"] = ind
    all_stacks.append(stack)

# -----------------------------
# 8. Joint pre-trend test: k = -1, -2
# -----------------------------
pretrend_results = []

for stack in all_stacks:
    ind = stack["industry"].iloc[0]

    pre_vars = []
    if "event_m1_dose" in stack.columns:
        pre_vars.append("event_m1_dose")
    if "event_m2_dose" in stack.columns:
        pre_vars.append("event_m2_dose")

    if len(pre_vars) < 2:
        continue

    formula = f"""
    employment_rate ~
        {' + '.join(pre_vars)}
        + dlogW
        + C(province) + C(date)
    """

    model = smf.ols(formula, data=stack).fit(
        cov_type="cluster",
        cov_kwds={"groups": stack["province"]}
    )

    hypotheses = " = 0, ".join(pre_vars) + " = 0"
    wald = model.wald_test(hypotheses, scalar=True)

    pretrend_results.append({
        "industry": ind,
        "pre_periods": "-1,-2",
        "wald_stat": float(wald.statistic),
        "p_value": float(wald.pvalue),
        "n_obs": int(model.nobs)
    })

pretrend_df = pd.DataFrame(pretrend_results).sort_values("p_value")

print("\n=== Joint Pre-trend Test (baseline = t0, k = -1, -2) ===")
print(pretrend_df.to_string(index=False))


Identified 2023+ MW increase events: 23

=== Joint Pre-trend Test (baseline = t0, k = -1, -2) ===
                                           industry pre_periods  wald_stat  p_value  n_obs
                  Health care and social assistance       -1,-2     6.6222   0.0365   4209
                          Services-producing sector       -1,-2     6.1532   0.0461   4209
    Professional, scientific and technical services       -1,-2     4.6326   0.0986   4209
                              Public administration       -1,-2     3.9027   0.1421   4209
                             Goods-producing sector       -1,-2     3.3005   0.1920   4209
      Other services (except public administration)       -1,-2     2.3083   0.3153   4209
                              Total, all industries       -1,-2     2.1731   0.3374   4209
                    Accommodation and food services       -1,-2     2.0963   0.3506   4209
                                      Manufacturing       -1,-2     2.0065   0.3667

Robustness Check 1: Testing whether the long-run effect decays by comparing event intervals above the median length versus those below the median length.

In [3]:
# ==========================================================
#  21 Industries: CLEAN stacked DDD Event Studies
#  + Long-run heterogeneity by inter-event gap (median split)
#  + F-test: H0 long_shortgap = long_longgap
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# -----------------------------
# 1. data upload and clean
# -----------------------------
base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
raw = pd.read_csv(fr"{base}\14100355.csv")

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"])

raw["province"] = raw["province"].astype(str).str.strip()
raw["industry"] = raw["industry"].astype(str).str.strip()

raw["min_wage"] = pd.to_numeric(raw["min_wage"], errors="coerce")
raw["employment_rate"] = pd.to_numeric(raw["employment_rate"], errors="coerce")

raw["m_id"] = raw["date"].dt.year * 12 + raw["date"].dt.month

# -----------------------------
# 2. pre-2023 wage_base
# -----------------------------
policy = raw.dropna(subset=["min_wage"]).copy()
policy = policy.sort_values(["province", "date"])

base_wage = (
    policy.loc[policy["date"] < "2023-01-01", ["province", "date", "min_wage"]]
    .sort_values(["province", "date"])
    .groupby("province", as_index=False)
    .tail(1)[["province", "min_wage"]]
    .rename(columns={"min_wage": "wage_base"})
)

raw = raw.merge(base_wage, on="province", how="left")
raw["dlogW"] = np.log(raw["min_wage"]) - np.log(raw["wage_base"])

# -----------------------------
# 3. Clean variable for regression
# -----------------------------
df = raw.dropna(subset=["employment_rate", "min_wage", "dlogW"]).copy()
df = df.sort_values(["province", "date"])

# -----------------------------
# 4. All of the events come from only 2023
# -----------------------------
df["dW"] = df.groupby("province")["min_wage"].diff()
df["lagW"] = df.groupby("province")["min_wage"].shift(1)
df["dose"] = (df["dW"] / df["lagW"]) * 100

events = (
    df.loc[
        (df["dose"] > 0) & (df["date"] >= "2023-01-01"),
        ["province", "date", "dose", "m_id"]
    ]
    .drop_duplicates()
    .rename(columns={"date": "t0", "m_id": "m0"})
    .reset_index(drop=True)
)

# ---------- inter-event gap ----------
events = events.sort_values(["province", "m0"]).reset_index(drop=True)
events["m0_next"] = events.groupby("province")["m0"].shift(-1)
events["gap_next"] = events["m0_next"] - events["m0"]

gap_median = events["gap_next"].dropna().median()
print(f"Median gap to next event (months): {gap_median}")

events["long_gap"] = (events["gap_next"] >= gap_median).astype(float)

print(f"Identified 2023+ MW increase events: {len(events)}")

# -----------------------------
# 5.  event window variables
# -----------------------------
L, R = 2, 3
BASE_K = -1
event_var = lambda k: f"event_{'m'+str(abs(k)) if k < 0 else 'p'+str(k)}_dose"

# -----------------------------
# 6. clean controls
# -----------------------------
df["in_any_window"] = 0
for prov, g in events.groupby("province"):
    m0_list = g["m0"].tolist()
    idx = df["province"].eq(prov)
    m = df.loc[idx, "m_id"].values
    hit = np.zeros_like(m, dtype=bool)
    for m0 in m0_list:
        hit |= (m >= m0 - L) & (m <= m0 + R)
    df.loc[idx, "in_any_window"] = hit.astype(int)

# -----------------------------
# 7. industry stacked DDD
# -----------------------------
industries = sorted(df["industry"].unique())
results = []

for ind in industries:
    dfi = df[df["industry"] == ind].copy()
    stack_list = []

    for ev in events.itertuples(index=False):
        ev_prov = ev.province
        m0 = ev.m0
        dose = ev.dose
        long_gap = ev.long_gap

        # ---- treated ----
        treated = dfi[dfi["province"] == ev_prov].copy()
        treated["EventTime"] = (treated["m_id"] - m0).astype(int)
        treated = treated[(treated["EventTime"] >= -L) &
                          (treated["EventTime"] <= R)].copy()

        # throw away overlap windows
        other_m0 = events.loc[
            (events["province"] == ev_prov) & (events["m0"] != m0),
            "m0"
        ].tolist()

        if other_m0:
            m = treated["m_id"].values
            overlap = np.zeros_like(m, dtype=bool)
            for om0 in other_m0:
                overlap |= (m >= om0 - L) & (m <= om0 + R)
            treated = treated[~overlap].copy()

        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            treated[event_var(k)] = ((treated["EventTime"] == k).astype(int)) * dose

        if treated.empty:
            continue

        treated["long_gap"] = long_gap

        # ---- controls ----
        controls = dfi[
            (dfi["province"] != ev_prov) &
            (dfi["in_any_window"] == 0)
        ].copy()

        for k in range(-L, R + 1):
            if k == BASE_K:
                continue
            controls[event_var(k)] = 0.0

        controls["long_gap"] = 0.0

        stack_list.append(pd.concat([treated, controls], ignore_index=True))

    if not stack_list:
        continue

    stack = pd.concat(stack_list, ignore_index=True)

    event_cols = [c for c in stack.columns if c.startswith("event_")]
    valid_events = [c for c in event_cols if stack[c].sum() != 0]
    if not valid_events:
        continue

    # ---- long-run heterogeneity ----
    stack["dlogW_shortgap"] = stack["dlogW"] * (1 - stack["long_gap"])
    stack["dlogW_longgap"] = stack["dlogW"] * stack["long_gap"]

    formula = f"""
    employment_rate ~
        dlogW_shortgap + dlogW_longgap
        + {' + '.join(valid_events)}
        + C(province) + C(date)
    """

    model = smf.ols(formula, data=stack).fit(
        cov_type="cluster",
        cov_kwds={"groups": stack["province"]}
    )

    # -----------------------------
    # ★ NEW: F/Wald test: H0 dlogW_shortgap = dlogW_longgap
    # -----------------------------
    try:
        ft = model.f_test("dlogW_shortgap = dlogW_longgap")
        p_f = float(ft.pvalue)
        f_stat = float(np.asarray(ft.fvalue).squeeze())
    except Exception:
        p_f = np.nan
        f_stat = np.nan

    # event term
    for v in valid_events:
        tag = v.split("_")[1]
        sign = tag[0]
        num = int(tag[1:])
        k = -num if sign == "m" else num
        results.append({
            "industry": ind,
            "k": k,
            "coef": model.params.get(v, np.nan),
            "p_value": model.pvalues.get(v, np.nan),
            "n_obs": int(model.nobs)
        })

    # long-run 
    results.append({
        "industry": ind,
        "k": "long_shortgap",
        "coef": model.params.get("dlogW_shortgap", np.nan),
        "p_value": model.pvalues.get("dlogW_shortgap", np.nan),
        "n_obs": int(model.nobs)
    })
    results.append({
        "industry": ind,
        "k": "long_longgap",
        "coef": model.params.get("dlogW_longgap", np.nan),
        "p_value": model.pvalues.get("dlogW_longgap", np.nan),
        "n_obs": int(model.nobs)
    })

    # 
    results.append({
        "industry": ind,
        "k": "Ftest_short_eq_long",
        "coef": f_stat,    
        "p_value": p_f,    
        "n_obs": int(model.nobs)
    })

# -----------------------------
# 8. Result output
# -----------------------------
res_df = pd.DataFrame(results).sort_values(["industry", "k"])

pd.set_option("display.max_rows", 600)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n=== CLEAN stacked DDD + long-run heterogeneity by inter-event gap (median split) ===")
print(res_df.to_string(index=False))

ft_only = res_df[res_df["k"] == "Ftest_short_eq_long"].sort_values("p_value")
print("\n=== F-test: H0 long_shortgap = long_longgap (sorted by p-value) ===")
print(ft_only.to_string(index=False))


Median gap to next event (months): 12.0
Identified 2023+ MW increase events: 23

=== CLEAN stacked DDD + long-run heterogeneity by inter-event gap (median split) ===
                                           industry                   k    coef  p_value  n_obs
                    Accommodation and food services                  -2 -0.0014   0.4183   4209
                    Accommodation and food services                   0  0.0005   0.5862   4209
                    Accommodation and food services                   1 -0.0002   0.8512   4209
                    Accommodation and food services                   2  0.0018   0.3546   4209
                    Accommodation and food services                   3  0.0003   0.8711   4209
                    Accommodation and food services Ftest_short_eq_long  0.0289   0.8687   4209
                    Accommodation and food services        long_longgap -0.1399   0.2627   4209
                    Accommodation and food services       long_sho

Robustness Check 2: A placebo (fake) shock to test short-run effects—placebo and true shocks yield clearly different effects.

In [2]:
# ==========================================================
#  CLEAN stacked DDD + Long-run ΔlogW
#  PLACEBO timing test (shift = -6 months) + JOINT
#  (robust to variable name pollution)
# ==========================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PLACEBO_SHIFT = -4  # could change to 6 or 8


L_PLACEBO = 2
R_PLACEBO = 3
BASE_K_PLACEBO = -1


# ---- build placebo events ----
placebo_events = events.copy()
placebo_events["m0"] = placebo_events["m0"] + PLACEBO_SHIFT
placebo_events["t0"] = placebo_events["t0"] + pd.DateOffset(months=PLACEBO_SHIFT)

print(f"Running placebo test with shift = {PLACEBO_SHIFT} months")

results_placebo = []

for ind in industries:
    dfi = df[df["industry"] == ind].copy()
    stack_list = []

    for ev in placebo_events.itertuples(index=False):

        treated = dfi[dfi["province"] == ev.province].copy()
        treated["EventTime"] = (treated["m_id"] - int(ev.m0)).astype(int)

        treated = treated[
            (treated["EventTime"] >= -L_PLACEBO) &
            (treated["EventTime"] <=  R_PLACEBO)
        ].copy()

        for k in range(-L_PLACEBO, R_PLACEBO + 1):
            if k == BASE_K_PLACEBO:
                continue
            treated[event_var(k)] = (treated["EventTime"].eq(k).astype(int) * ev.dose)

        if treated.empty:
            continue

        controls = dfi[
            (dfi["province"] != ev.province) &
            (dfi["in_any_window"] == 0)
        ].copy()

        for k in range(-L_PLACEBO, R_PLACEBO + 1):
            if k == BASE_K_PLACEBO:
                continue
            controls[event_var(k)] = 0.0

        stack_list.append(pd.concat([treated, controls], ignore_index=True))

    if not stack_list:
        continue

    stack = pd.concat(stack_list, ignore_index=True)

    # 长期项：仍然用 out-of-window（跟你主回归口径一致）
    stack["dlogW_out"] = stack["dlogW"] * (1 - stack["in_any_window"])

    event_cols = [c for c in stack.columns if c.startswith("event_")]
    valid_events = [c for c in event_cols if stack[c].sum() != 0]
    if not valid_events:
        continue

    formula = f"""
        employment_rate ~
            dlogW_out
            + {' + '.join(valid_events)}
            + C(province) + C(date)
    """

    model = smf.ols(formula, data=stack).fit(
        cov_type="cluster",
        cov_kwds={"groups": stack["province"]}
    )

    # k-by-k
    for v in valid_events:
        tag = v.split("_")[1]  # m2 or p3
        sign = tag[0]
        num = int(tag[1:])
        kk = -num if sign == "m" else num

        results_placebo.append({
            "industry": ind,
            "k": kk,
            "coef": model.params.get(v, np.nan),
            "p_value": model.pvalues.get(v, np.nan),
            "n_obs": int(model.nobs)
        })

    # JOINT Wald test (scalar=True avoids FutureWarning + numpy warning)
    param_names = model.params.index.tolist()
    Rmat = np.zeros((len(valid_events), len(param_names)))

    for i, v in enumerate(valid_events):
        if v in param_names:
            Rmat[i, param_names.index(v)] = 1

    wald = model.wald_test(Rmat, scalar=True)

    results_placebo.append({
        "industry": ind,
        "k": "JOINT",
        "coef": float(wald.statistic),
        "p_value": float(wald.pvalue),
        "n_obs": int(model.nobs)
    })

placebo_df = pd.DataFrame(results_placebo).sort_values(["industry", "k"])

print("\n=== PLACEBO stacked DDD + JOINT results ===")
print(placebo_df.to_string(index=False))

OUT_PLACEBO = r"C:\Users\SC2zh\Desktop\S4 paper\results_placebo_ddd.csv"
placebo_df.to_csv(OUT_PLACEBO, index=False)
print(f"\nPlacebo results saved to: {OUT_PLACEBO}")


Running placebo test with shift = -4 months

=== PLACEBO stacked DDD + JOINT results ===
                                           industry     k    coef  p_value  n_obs
                    Accommodation and food services    -2  0.0008   0.2033   4194
                    Accommodation and food services     0 -0.0004   0.5249   4194
                    Accommodation and food services     1 -0.0005   0.0632   4194
                    Accommodation and food services     2 -0.0019   0.2177   4194
                    Accommodation and food services     3 -0.0021   0.0920   4194
                    Accommodation and food services JOINT 15.7117   0.0077   4194
                                        Agriculture    -2 -0.0020   0.3140   4194
                                        Agriculture     0  0.0010   0.5251   4194
                                        Agriculture     1 -0.0026   0.0195   4194
                                        Agriculture     2  0.0016   0.7149   4194
         

In [24]:
main = pd.read_csv(
    r"C:\Users\SC2zh\Desktop\S4 paper\results_main_ddd.csv"
)

placebo = pd.read_csv(
    r"C:\Users\SC2zh\Desktop\S4 paper\results_placebo_ddd.csv"
)

compare_joint = (
    main.query("k == 'JOINT'")
    .merge(
        placebo.query("k == 'JOINT'"),
        on="industry",
        suffixes=("_main", "_placebo")
    )
    .sort_values("p_value_main")
)

print("\n=== Main vs Placebo: JOINT Wald tests ===")
print(compare_joint.to_string(index=False))



=== Main vs Placebo: JOINT Wald tests ===
                                           industry k_main  coef_main  p_value_main  n_obs_main k_placebo  coef_placebo  p_value_placebo  n_obs_placebo
                    Accommodation and food services  JOINT    48.6972        0.0000        4209     JOINT       15.7117           0.0077           4194
                                      Manufacturing  JOINT    22.1617        0.0005        4209     JOINT        3.9092           0.5626           4194
                                        Agriculture  JOINT    20.9340        0.0008        4209     JOINT       19.9505           0.0013           4194
                              Public administration  JOINT    18.7041        0.0022        4209     JOINT        5.4658           0.3617           4194
                  Health care and social assistance  JOINT    18.6329        0.0022        4209     JOINT       13.8319           0.0167           4194
    Professional, scientific and technical se

Robustness Check 3: conducts permutation inference for the long-run minimum-wage effect by randomly shuffling Δlog(min wage) across provinces within each date to assess whether the estimated long-run coefficients are distinguishable from chance.

In [3]:
# ==========================================================
#  CLEAN stacked DDD + Long-run ΔlogW
#  Permutation inference for LONG effect (dlogW)
#  Permute dlogW across provinces WITHIN each date
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# -----------------------------
# 0) Parameters
# -----------------------------
L, R = 2, 3
BASE_K = -1
B = 100              # Number of permutations: start with 200; use 500/1000 for more stability
SEED = 123
event_var = lambda k: f"event_{'m'+str(abs(k)) if k < 0 else 'p'+str(k)}_dose"

# -----------------------------
# 1) Read & clean data (consistent with main specification)
# -----------------------------
base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
raw = pd.read_csv(fr"{base}\14100355.csv")

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"])

raw["province"] = raw["province"].astype(str).str.strip()
raw["industry"] = raw["industry"].astype(str).str.strip()

raw["min_wage"] = pd.to_numeric(raw["min_wage"], errors="coerce")
raw["employment_rate"] = pd.to_numeric(raw["employment_rate"], errors="coerce")

raw["m_id"] = raw["date"].dt.year * 12 + raw["date"].dt.month

# wage_base constructed from policy-only rows
policy = raw.dropna(subset=["min_wage"]).copy().sort_values(["province", "date"])
base_wage = (
    policy.loc[policy["date"] < "2023-01-01", ["province", "date", "min_wage"]]
          .sort_values(["province", "date"])
          .groupby("province", as_index=False)
          .tail(1)[["province", "min_wage"]]
          .rename(columns={"min_wage": "wage_base"})
)

raw = raw.merge(base_wage, on="province", how="left")
raw["dlogW"] = np.log(raw["min_wage"]) - np.log(raw["wage_base"])

# Regression dataframe
df = raw.dropna(subset=["employment_rate", "min_wage", "dlogW"]).copy()
df = df.sort_values(["province", "date"])

# -----------------------------
# 2) Identify 2023+ events (consistent with main specification)
# -----------------------------
df["dW"] = df.groupby("province")["min_wage"].diff()
df["lagW"] = df.groupby("province")["min_wage"].shift(1)
df["dose"] = (df["dW"] / df["lagW"]) * 100

events = (
    df.loc[(df["dose"] > 0) & (df["date"] >= "2023-01-01"),
           ["province", "date", "dose", "m_id"]]
      .drop_duplicates()
      .rename(columns={"date": "t0", "m_id": "m0"})
      .reset_index(drop=True)
)

print(f"Identified 2023+ MW increase events: {len(events)}")

# -----------------------------
# 3) Clean-controls indicator (consistent with main specification)
# -----------------------------
df["in_any_window"] = 0
for prov, g in events.groupby("province"):
    m0_list = g["m0"].tolist()
    idx = df["province"].eq(prov)
    m = df.loc[idx, "m_id"].values
    hit = np.zeros_like(m, dtype=bool)
    for m0 in m0_list:
        hit |= (m >= m0 - L) & (m <= m0 + R)
    df.loc[idx, "in_any_window"] = hit.astype(int)

industries = sorted(df["industry"].unique())

# -----------------------------
# 4) Helper function: run industry-level LONG regression
#    given a dlogW column name
# -----------------------------
def run_long_by_ind(dlog_col: str) -> pd.DataFrame:
    out = []

    for ind in industries:
        dfi = df[df["industry"] == ind].copy()
        stack_list = []

        for ev in events.itertuples(index=False):
            ev_prov = ev.province
            m0 = ev.m0
            dose = ev.dose

            treated = dfi[dfi["province"] == ev_prov].copy()
            treated["EventTime"] = (treated["m_id"] - m0).astype(int)
            treated = treated[(treated["EventTime"] >= -L) & (treated["EventTime"] <= R)].copy()

            # Drop overlapping event windows from other events in the same province
            other_m0 = events.loc[(events["province"] == ev_prov) & (events["m0"] != m0), "m0"].tolist()
            if other_m0:
                m = treated["m_id"].values
                overlap = np.zeros_like(m, dtype=bool)
                for om0 in other_m0:
                    overlap |= (m >= om0 - L) & (m <= om0 + R)
                treated = treated[~overlap].copy()

            for k in range(-L, R + 1):
                if k == BASE_K:
                    continue
                treated[event_var(k)] = (treated["EventTime"] == k).astype(int) * dose

            if treated.empty:
                continue

            controls = dfi[(dfi["province"] != ev_prov) & (dfi["in_any_window"] == 0)].copy()
            for k in range(-L, R + 1):
                if k == BASE_K:
                    continue
                controls[event_var(k)] = 0.0

            stack_list.append(pd.concat([treated, controls], ignore_index=True))

        if not stack_list:
            continue

        stack = pd.concat(stack_list, ignore_index=True)

        event_cols = [c for c in stack.columns if c.startswith("event_")]
        valid_events = [c for c in event_cols if stack[c].sum() != 0]
        if not valid_events:
            continue

        formula = f"""
        employment_rate ~
            {dlog_col}
            + {' + '.join(valid_events)}
            + C(province) + C(date)
        """

        m = smf.ols(formula, data=stack).fit(
            cov_type="cluster",
            cov_kwds={"groups": stack["province"]}
        )

        out.append({
            "industry": ind,
            "beta_long": m.params.get(dlog_col, np.nan),
            "p_long_asym": m.pvalues.get(dlog_col, np.nan),  # For reference only (do not over-interpret with 10 clusters)
            "n_obs": int(m.nobs)
        })

    return pd.DataFrame(out)

# -----------------------------
# 5) Observed LONG effects
# -----------------------------
obs = run_long_by_ind("dlogW").sort_values("industry").reset_index(drop=True)
print("\n=== Observed LONG (dlogW) ===")
print(obs.to_string(index=False, float_format=lambda x: f"{x: .4f}"))

# -----------------------------
# 6) Permutation: shuffle dlogW across provinces within each date
# -----------------------------
rng = np.random.default_rng(SEED)

perm_betas = {ind: [] for ind in obs["industry"].tolist()}

for b in range(B):
    df["dlogW_perm"] = df.groupby("date")["dlogW"].transform(
        lambda s: pd.Series(rng.permutation(s.values), index=s.index)
    )

    perm = run_long_by_ind("dlogW_perm").set_index("industry")

    # Record results only for industries present in obs
    for ind in perm_betas.keys():
        if ind in perm.index:
            perm_betas[ind].append(perm.loc[ind, "beta_long"])
        else:
            perm_betas[ind].append(np.nan)

# -----------------------------
# 7) Permutation p-values: Pr(|beta_perm| >= |beta_obs|)
# -----------------------------
rows = []
for _, r in obs.iterrows():
    ind = r["industry"]
    b_obs = r["beta_long"]
    draws = np.array(perm_betas[ind], dtype=float)
    draws = draws[~np.isnan(draws)]

    if len(draws) == 0 or np.isnan(b_obs):
        p_perm = np.nan
    else:
        p_perm = np.mean(np.abs(draws) >= np.abs(b_obs))

    rows.append({
        "industry": ind,
        "beta_long_obs": b_obs,
        "p_long_perm": p_perm,
        "B_effective": int(len(draws)),
        "n_obs": int(r["n_obs"])
    })

perm_summary = pd.DataFrame(rows).sort_values("p_long_perm")
print("\n=== Permutation inference for LONG (within-date shuffle) ===")
print(perm_summary.to_string(index=False, float_format=lambda x: f"{x: .4f}"))


Identified 2023+ MW increase events: 23

=== Observed LONG (dlogW) ===
                                           industry  beta_long  p_long_asym  n_obs
                    Accommodation and food services    -0.1204       0.0091   4209
                                        Agriculture     0.2667       0.2742   4209
      Business, building and other support services     0.0525       0.6356   4209
                                       Construction     0.0056       0.9525   4209
                               Educational services    -0.1007       0.1625   4209
Finance, insurance, real estate, rental and leasing    -0.0638       0.5178   4209
  Forestry, fishing, mining, quarrying, oil and gas     0.2173       0.4088   4209
                             Goods-producing sector    -0.0244       0.7871   4209
                  Health care and social assistance    -0.0270       0.6336   4209
                Information, culture and recreation     0.0103       0.8826   4209
                

Robustness Check 4: Spillver effect check

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# =========================
# 0) Parameters
# =========================
base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
DATA = fr"{base}\14100355.csv"

USE_NEIGHBOR_SPILLOVER = True   # True: neighboring-province spillovers; False: national-average spillovers

# =========================
# 1) Read data & construct dlogW (consistent with main specification)
# =========================
raw = pd.read_csv(DATA)

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"])

raw["province"] = raw["province"].astype(str).str.strip()
raw["industry"] = raw["industry"].astype(str).str.strip()

raw["min_wage"] = pd.to_numeric(raw["min_wage"], errors="coerce")
raw["employment_rate"] = pd.to_numeric(raw["employment_rate"], errors="coerce")

# wage_base: last observed minimum wage before 2023-01 for each province
policy = raw.dropna(subset=["min_wage"]).copy().sort_values(["province", "date"])
base_wage = (
    policy.loc[policy["date"] < "2023-01-01", ["province", "date", "min_wage"]]
          .sort_values(["province", "date"])
          .groupby("province", as_index=False)
          .tail(1)[["province", "min_wage"]]
          .rename(columns={"min_wage": "wage_base"})
)

raw = raw.merge(base_wage, on="province", how="left")
raw["dlogW"] = np.log(raw["min_wage"]) - np.log(raw["wage_base"])

# Regression sample
# (employment_rate is only available from 2023 onward, as discussed)
df = raw.dropna(subset=["employment_rate", "dlogW"]).copy()

# =========================
# 2) Construct spillover variables
# =========================
provs = sorted(df["province"].unique())

# ---- Adjacency list (can be modified/refined if needed) ----
neighbors = {
    "British Columbia": ["Alberta"],
    "Alberta": ["British Columbia", "Saskatchewan"],
    "Saskatchewan": ["Alberta", "Manitoba"],
    "Manitoba": ["Saskatchewan", "Ontario"],
    "Ontario": ["Manitoba", "Quebec"],
    "Quebec": ["Ontario", "New Brunswick"],
    "New Brunswick": ["Quebec", "Nova Scotia", "Prince Edward Island"],
    "Nova Scotia": ["New Brunswick", "Prince Edward Island"],
    "Prince Edward Island": ["New Brunswick", "Nova Scotia"],
    "Newfoundland and Labrador": ["Nova Scotia"],  # optional: can also be set to an empty list
}

# Province-date table of dlogW
wtab = df[["province", "date", "dlogW"]].drop_duplicates()

# National-average spillover (excluding own province)
# spill_all_{p,t} = mean_{q != p} dlogW_{q,t}
w_pivot = wtab.pivot(index="date", columns="province", values="dlogW").sort_index()

spill_all = {}
for p in provs:
    others = [q for q in provs if q != p]
    spill_all[p] = w_pivot[others].mean(axis=1)

spill_all_df = pd.DataFrame(spill_all).stack().reset_index()
spill_all_df.columns = ["date", "province", "spill_dlogW_all"]

# Neighbor-province average spillover
spill_nb = {}
for p in provs:
    nb = neighbors.get(p, [])
    if len(nb) == 0:
        # If no neighbors, set to NaN (will be dropped in regressions),
        # alternatively could be set to zero
        spill_nb[p] = pd.Series(np.nan, index=w_pivot.index)
    else:
        spill_nb[p] = w_pivot[nb].mean(axis=1)

spill_nb_df = pd.DataFrame(spill_nb).stack().reset_index()
spill_nb_df.columns = ["date", "province", "spill_dlogW_nb"]

# Merge spillovers back to df
df = df.merge(spill_all_df, on=["date", "province"], how="left")
df = df.merge(spill_nb_df, on=["date", "province"], how="left")

# Choose which spillover measure to use
df["spill_dlogW"] = df["spill_dlogW_nb"] if USE_NEIGHBOR_SPILLOVER else df["spill_dlogW_all"]

# =========================
# 3) Industry-level TWFE regressions: own + spillover
# =========================
results = []
industries = sorted(df["industry"].unique())

for ind in industries:
    dfi = df[df["industry"] == ind].copy()

    # Spillover may be missing (e.g., provinces with no defined neighbors)
    dfi = dfi.dropna(subset=["employment_rate", "dlogW", "spill_dlogW"])

    if dfi.empty:
        continue

    # Cluster standard errors at the province level
    formula = """
    employment_rate ~ dlogW + spill_dlogW + C(province) + C(date)
    """

    m = smf.ols(formula, data=dfi).fit(
        cov_type="cluster",
        cov_kwds={"groups": dfi["province"]}
    )

    results.append({
        "industry": ind,
        "beta_own": m.params.get("dlogW", np.nan),
        "p_own": m.pvalues.get("dlogW", np.nan),
        "beta_spill": m.params.get("spill_dlogW", np.nan),
        "p_spill": m.pvalues.get("spill_dlogW", np.nan),
        "n_obs": int(m.nobs)
    })

res = pd.DataFrame(results).sort_values("p_spill")
print("\n=== Spillover test (own dlogW + spill dlogW) ===")
print(res.to_string(index=False, float_format=lambda x: f"{x: .4f}"))



=== Spillover test (own dlogW + spill dlogW) ===
                                           industry  beta_own   p_own  beta_spill  p_spill  n_obs
                     Transportation and warehousing   -0.2712  0.0083      0.3695   0.0004    330
  Forestry, fishing, mining, quarrying, oil and gas    0.2389  0.3103      0.8433   0.0022    330
    Professional, scientific and technical services   -0.2389  0.0281      0.2614   0.0831    330
      Business, building and other support services    0.1522  0.2178     -0.3004   0.0926    330
                    Accommodation and food services   -0.0638  0.4186      0.1658   0.2402    330
                                          Utilities   -0.0754  0.6543      0.3458   0.2418    326
                          Services-producing sector   -0.0204  0.5278      0.0518   0.2603    330
                              Total, all industries   -0.0019  0.9610      0.0486   0.3365    330
Finance, insurance, real estate, rental and leasing    0.0807  0.460

In [27]:
# ==========================================================
# CHECK 1: Correlation between own dlogW and spill_dlogW
# ==========================================================

corr_rows = []

for ind in industries:
    dfi = df[df["industry"] == ind].copy()
    dfi = dfi.dropna(subset=["dlogW", "spill_dlogW"])

    if len(dfi) < 10:
        continue

    corr = dfi["dlogW"].corr(dfi["spill_dlogW"])

    corr_rows.append({
        "industry": ind,
        "corr_own_spill": corr,
        "n_obs": len(dfi)
    })

corr_df = (
    pd.DataFrame(corr_rows)
      .sort_values("corr_own_spill", ascending=False)
)

print("\n=== Correlation between own dlogW and spill_dlogW ===")
print(corr_df.to_string(index=False, float_format=lambda x: f"{x: .3f}"))



=== Correlation between own dlogW and spill_dlogW ===
                                           industry  corr_own_spill  n_obs
                    Accommodation and food services           0.591    330
                                        Agriculture           0.591    330
                     Transportation and warehousing           0.591    330
                              Total, all industries           0.591    330
                          Services-producing sector           0.591    330
                              Public administration           0.591    330
    Professional, scientific and technical services           0.591    330
      Other services (except public administration)           0.591    330
                                      Manufacturing           0.591    330
                Information, culture and recreation           0.591    330
                  Health care and social assistance           0.591    330
                             Goods-producing 

In [5]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# =========================
# 0) Paths
# =========================
base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
DATA = fr"{base}\14100355.csv"

# =========================
# 1) Read data & construct dlogW
# =========================
raw = pd.read_csv(DATA)

raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = raw.dropna(subset=["date"])

raw["province"] = raw["province"].astype(str).str.strip()
raw["industry"] = raw["industry"].astype(str).str.strip()

raw["min_wage"] = pd.to_numeric(raw["min_wage"], errors="coerce")
raw["employment_rate"] = pd.to_numeric(raw["employment_rate"], errors="coerce")

# ---- wage_base (last observation before 2023-01 for each province) ----
policy = raw.dropna(subset=["min_wage"]).copy().sort_values(["province", "date"])
base_wage = (
    policy.loc[policy["date"] < "2023-01-01", ["province", "date", "min_wage"]]
          .groupby("province", as_index=False)
          .tail(1)[["province", "min_wage"]]
          .rename(columns={"min_wage": "wage_base"})
)

raw = raw.merge(base_wage, on="province", how="left")
raw["dlogW"] = np.log(raw["min_wage"]) - np.log(raw["wage_base"])

df = raw.dropna(subset=["employment_rate", "dlogW"]).copy()

# =========================
# 2) Define Near / Far provinces
# =========================
NEAR_PROVINCES = {
    "British Columbia": ["Alberta"],
    "Alberta": ["British Columbia", "Saskatchewan"],
    "Saskatchewan": ["Alberta", "Manitoba"],
    "Manitoba": ["Saskatchewan", "Ontario"],
    "Ontario": ["Manitoba", "Quebec"],
    "Quebec": ["Ontario", "New Brunswick"],
    "New Brunswick": ["Quebec", "Nova Scotia", "Prince Edward Island"],
    "Nova Scotia": ["New Brunswick", "Prince Edward Island"],
    "Prince Edward Island": ["New Brunswick", "Nova Scotia"],
    "Newfoundland and Labrador": ["Nova Scotia"]
}

ALL_PROVINCES = sorted(df["province"].unique())

FAR_PROVINCES = {
    p: [q for q in ALL_PROVINCES if q != p and q not in NEAR_PROVINCES.get(p, [])]
    for p in ALL_PROVINCES
}

# =========================
# 3) Construct Near / Far spillovers (FIXED)
# =========================
wtab = (
    df[["province", "date", "dlogW"]]
    .drop_duplicates()
    .pivot(index="date", columns="province", values="dlogW")
    .sort_index()
)

# ---- Near spillover ----
spill_near = {}
for p in ALL_PROVINCES:
    nbs = NEAR_PROVINCES.get(p, [])
    if len(nbs) == 0:
        spill_near[p] = pd.Series(np.nan, index=wtab.index)
    else:
        spill_near[p] = wtab[nbs].mean(axis=1)

spill_near_df = (
    pd.DataFrame(spill_near)
      .stack()
      .reset_index()
      .rename(columns={
          "level_0": "date",
          "level_1": "province",
          0: "spill_dlogW_near"
      })
)

# ---- Far spillover ----
spill_far = {}
for p in ALL_PROVINCES:
    fars = FAR_PROVINCES[p]
    spill_far[p] = wtab[fars].mean(axis=1)

spill_far_df = (
    pd.DataFrame(spill_far)
      .stack()
      .reset_index()
      .rename(columns={
          "level_0": "date",
          "level_1": "province",
          0: "spill_dlogW_far"
      })
)

# Merge back to df
df = df.merge(spill_near_df, on=["date", "province"], how="left")
df = df.merge(spill_far_df, on=["date", "province"], how="left")
print(df[["province", "date", "dlogW", "spill_dlogW_near", "spill_dlogW_far"]].head())

# =========================
# 4) Industry-level TWFE regressions
# =========================
results = []

for ind in sorted(df["industry"].unique()):
    dfi = df[df["industry"] == ind].copy()
    dfi = dfi.dropna(subset=["spill_dlogW_near", "spill_dlogW_far"])

    if dfi.empty:
        continue

    formula = """
    employment_rate ~
        dlogW
      + spill_dlogW_near
      + spill_dlogW_far
      + C(province) + C(date)
    """

    m = smf.ols(formula, data=dfi).fit(
        cov_type="cluster",
        cov_kwds={"groups": dfi["province"]}
    )

    results.append({
        "industry": ind,
        "beta_own": m.params.get("dlogW", np.nan),
        "p_own": m.pvalues.get("dlogW", np.nan),
        "beta_near": m.params.get("spill_dlogW_near", np.nan),
        "p_near": m.pvalues.get("spill_dlogW_near", np.nan),
        "beta_far": m.params.get("spill_dlogW_far", np.nan),
        "p_far": m.pvalues.get("spill_dlogW_far", np.nan),
        "n_obs": int(m.nobs)
    })

res = pd.DataFrame(results).sort_values("p_near")

print("\n=== Near vs Far Spillover Test ===")
print(res.to_string(index=False, float_format=lambda x: f"{x: .4f}"))


  province       date  dlogW  spill_dlogW_near  spill_dlogW_far
0  Alberta 2023-01-01 0.0000            0.0000           0.0081
1  Alberta 2023-02-01 0.0000            0.0000           0.0081
2  Alberta 2023-03-01 0.0000            0.0000           0.0081
3  Alberta 2023-04-01 0.0000            0.0000           0.0421
4  Alberta 2023-05-01 0.0000            0.0000           0.0518

=== Near vs Far Spillover Test ===
                                           industry  beta_own   p_own  beta_near  p_near  beta_far   p_far  n_obs
                Information, culture and recreation   -0.8582  0.0163    -0.8965  0.0000   -4.3448  0.0020    330
                          Services-producing sector    0.0322  0.6324     0.1298  0.0040    0.3672  0.2234    330
                                      Manufacturing   -0.3030  0.0379    -0.3685  0.0574   -1.8466  0.0393    330
                               Educational services    0.1046  0.4800     0.1682  0.1413    0.5130  0.4184    330
          